In [1]:
# load libraries
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report as sklearn_classification_report
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from pathlib import Path
import sys

# set path to project root and import custom functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.classification import train_bert
from utils.evaluation import run_testset_stance

In [2]:
# load the data
with open("../../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# subset to only data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into a training and a validation set
train_data, val_data = train_test_split(data_with_annotations, test_size=0.2, shuffle=True, random_state=42)

### StanceBERTa
This model is DistilBERT-based model for sequence classification which finds the predominant entity in a text and predicts the stance towards it. To make it suitable to my case I need to set the social group at the start of the text, which is not ideal and makes the task very similar to a sequence classification.\
The results are not better than the ones obtained with the sequence classification setup.

In [7]:
# finetune stanceberta on my dataset and evaluate directly
class StanceSpanDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_len, label2id):
        self.dataset = []
        for item in raw_data:
            sentence = item['sentence']
            for ann in item.get('annotations', []):
                span_text = ann['text']

                # format input for the model: "[span]: [sentence]"
                input_text = f"{span_text}: {sentence}"

                # get the stance label
                stance_label = label2id[ann['tag'].lower()[3:]]

                # tokenize the input
                encoding = tokenizer(
                    input_text,
                    truncation=True,
                    padding='max_length',
                    max_length=max_len,
                    return_tensors='pt'
                )
                self.dataset.append({
                    "input_ids": encoding["input_ids"].squeeze(0),
                    "attention_mask": encoding["attention_mask"].squeeze(0),
                    "label": torch.tensor(stance_label, dtype=torch.long)
                })

    

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
       return self.dataset[idx]

# set model name, get config and label2id mapping  
model_name = "eevvgg/StanceBERTa"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_config = AutoConfig.from_pretrained(model_name)
model_label_to_id = model_config.label2id 

# build my custom label2id mapping
tag_to_model_label = {
    'neg': 'negative',
    'neutral': 'neutral',
    'pos': 'positive'
}
label_to_id = {tag: model_label_to_id[model_label] for tag, model_label in tag_to_model_label.items()}
id_to_label = {v: k for k, v in label_to_id.items()}

# create training and validation dataset and dataloaders
train_dataset = StanceSpanDataset(raw_data=train_data, tokenizer=tokenizer, max_len=128, label2id=label_to_id)
validation_dataset = StanceSpanDataset(raw_data=val_data, tokenizer=tokenizer, max_len=128, label2id=label_to_id)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=16, shuffle=False)

# create model and optimizer
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
optimizer = AdamW(model.parameters(), lr = 5e-5, weight_decay=0.01)
epochs = 10

# finetune on the downstream task
train_bert(train_loader, model, optimizer, epochs, device, "stance")

# evaluate the model on the validation/test set
true_labels, pred_labels, _ = run_testset_stance(model, val_loader, device)

print(sklearn_classification_report(
    [id_to_label[i] for i in true_labels],
    [id_to_label[i] for i in pred_labels]
    ))

Epoch 1/10


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Training: 100%|██████████| 103/103 [00:23<00:00,  4.40it/s, loss=0.748]


Average training loss: 0.7414
Epoch 2/10


Training: 100%|██████████| 103/103 [00:22<00:00,  4.57it/s, loss=0.18] 


Average training loss: 0.4548
Epoch 3/10


...

Average training loss: 0.0946
Epoch 9/10


Training: 100%|██████████| 103/103 [00:22<00:00,  4.54it/s, loss=0.173]   


Average training loss: 0.0929
Epoch 10/10


Training: 100%|██████████| 103/103 [00:22<00:00,  4.52it/s, loss=0.334]  


Average training loss: 0.0952
              precision    recall  f1-score   support

         neg       0.86      0.33      0.48        18
     neutral       0.68      0.46      0.55       138
         pos       0.73      0.89      0.80       259

    accuracy                           0.72       415
   macro avg       0.76      0.56      0.61       415
weighted avg       0.72      0.72      0.71       415



### Stance Detection SemEval

This model is a BERTweet-finetuned (trained like RoBERTa) model for sequence classification which was trained on the SemEval2016-Task6 stance detection dataset. It outputs FAVOR, AGAINST and NONE which can be translated to my case quite well. In the original dataset, the entitiy does not need to actually be part of the input text.\
The results point into a similar direction as the ones before, but they are actually worse.

In [12]:
# finetune stanceberta on my dataset and evaluate directly
class StanceSpanDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_len, label2id):
        self.dataset = []

        for item in raw_data:
            sentence = item['sentence']
            for ann in item.get('annotations', []):
                span_text = ann['text']
                stance_label = label2id[ann['tag'].lower()[3:]]

                encoding = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding='max_length',
                    max_length=max_len,
                    return_tensors='pt'
                )
                self.dataset.append({
                    "input_ids": encoding["input_ids"].squeeze(0),
                    "attention_mask": encoding["attention_mask"].squeeze(0),
                    "label": torch.tensor(stance_label, dtype=torch.long)
                })

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

model_name = "krishnagarg09/stance-detection-semeval2016"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_config = AutoConfig.from_pretrained(model_name)
id2label = model_config.id2label 
tag_to_model_label = {
    'neg': 'AGAINST',
    'neutral':'NONE',
    'pos': 'FAVOR'
}

label_to_id = { tag: 
                  next(idx for idx, lbl in id2label.items() if lbl == model_label)
                for tag, model_label in tag_to_model_label.items() }
id_to_label = {v: k for k, v in label_to_id.items()}

# create training and validation dataset and dataloaders
train_dataset = StanceSpanDataset(raw_data=train_data, tokenizer=tokenizer, max_len=128, label2id=label_to_id)
validation_dataset = StanceSpanDataset(raw_data=val_data, tokenizer=tokenizer, max_len=128, label2id=label_to_id)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=16, shuffle=False)

# create model and optimizer
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
optimizer = AdamW(model.parameters(), lr = 5e-5, weight_decay=0.01)
epochs = 10

# finetune on the downstream task
train_bert(train_loader, model, optimizer, epochs, device, "stance")

# evaluate the model on the validation/test set
true_labels, pred_labels, _ = run_testset_stance(model, val_loader, device)

print(sklearn_classification_report(
    [id_to_label[i] for i in true_labels],
    [id_to_label[i] for i in pred_labels]
    ))

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Epoch 1/10


Training: 100%|██████████| 103/103 [00:43<00:00,  2.36it/s, loss=0.679]


Average training loss: 0.7842
Epoch 2/10


Training: 100%|██████████| 103/103 [00:42<00:00,  2.40it/s, loss=0.707]


...

Average training loss: 0.1558
Epoch 9/10


Training: 100%|██████████| 103/103 [00:42<00:00,  2.43it/s, loss=0.0495] 


Average training loss: 0.1397
Epoch 10/10


Training: 100%|██████████| 103/103 [00:42<00:00,  2.42it/s, loss=0.0101] 


Average training loss: 0.1076
              precision    recall  f1-score   support

         neg       0.58      0.78      0.67        18
     neutral       0.57      0.57      0.57       138
         pos       0.78      0.76      0.77       259

    accuracy                           0.70       415
   macro avg       0.64      0.70      0.67       415
weighted avg       0.70      0.70      0.70       415



### NLI Stance Model (Howe et al.)

The model is based on the NLI model developed by Moritz Laurer (2024). It is a DeBERTa model finetuned on NLI. The authors then applied further transfer learning to finetune it to evaluate hypotheses on stances towards social groups mentioned in text.

#### Off-the-Shelf Predictions
First I test how well the already pretrained model of the authors works without any further finetuning to the domain of parliamentary questions. It does not work any better than the other models so far.

In [14]:
class StanceNLIDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_len, label2id):
        self.dataset = []

        for item in raw_data:
            sentence = item["sentence"]
            for ann in item.get("annotations", []):
                target = ann["text"]
                gold_stance = ann["tag"].lower()[3:]

                hypotheses = {
                    "pos": f"The text is positive towards {target}.",
                    "neg": f"The text is negative towards {target}.",
                    "neutral": f"The text is neutral, or contains no stance, towards {target}."
                }

                for stance, hypothesis in hypotheses.items():
                    label_text = "entailment" if stance == gold_stance else "contradiction"

                    encoding = tokenizer(
                        sentence,
                        hypothesis,
                        truncation=True,
                        padding="max_length",
                        max_length=max_len,
                        return_tensors="pt"
                    )
                    self.dataset.append({
                        "input_ids": encoding["input_ids"].squeeze(0),
                        "attention_mask": encoding["attention_mask"].squeeze(0),
                        "label": label2id[label_text]
                    })
                

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

model_name = "rwillh11/mdeberta_NLI_stance_NoContext"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def evaluate_nli_stance(model, data):
    model.eval()
    all_preds = []
    all_labels = []

    for item in data:
        sentence = item["sentence"]
        for ann in item.get("annotations", []):
            target = ann["text"]
            gold_stance = ann["tag"].lower()[3:]

            hypotheses = {
                "pos": f"The text is positive towards {target}.",
                "neg": f"The text is negative towards {target}.",
                "neutral": f"The text is neutral, or contains no stance, towards {target}."
            }

            # Tokenize all 3 hypotheses as a batch
            inputs = tokenizer(
                [sentence]*3,
                list(hypotheses.values()),
                return_tensors="pt",
                padding=True,
                truncation=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                entail_probs = probs[:, 0].tolist() # 0 is the entailment index

            # choose hypothesis with highest entailment probability
            predicted_stance = list(hypotheses.keys())[entail_probs.index(max(entail_probs))]

            all_preds.append(predicted_stance)
            all_labels.append(gold_stance)

    return all_labels, all_preds


true_labels, pred_labels = evaluate_nli_stance(model, val_data)
print(sklearn_classification_report(true_labels, pred_labels))


              precision    recall  f1-score   support

         neg       0.52      0.61      0.56        18
     neutral       0.57      0.44      0.50       138
         pos       0.73      0.81      0.77       259

    accuracy                           0.68       415
   macro avg       0.61      0.62      0.61       415
weighted avg       0.67      0.68      0.67       415



Now I turn to applying further transfer learning by finetuning it to my dataset.

In [18]:
# create model and optimizer
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
optimizer = AdamW(model.parameters(), lr = 5e-5, weight_decay=0.01)
epochs = 10

label_to_id = model.config.label2id
train_dataset = StanceNLIDataset(train_data, tokenizer, max_len=128, label2id=label_to_id)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = StanceNLIDataset(val_data, tokenizer, max_len=128, label2id=label_to_id)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


# finetune on the downstream task
train_bert(train_loader, model, optimizer, epochs, device, "stance")

true_labels, pred_labels = evaluate_nli_stance(model, val_data)
print(sklearn_classification_report(true_labels, pred_labels))

Epoch 1/10


Training:   6%|▌         | 18/309 [01:27<23:41,  4.88s/it, loss=0.964]


KeyboardInterrupt: 